# TP 01 — Construire un moteur de recherche sémantique

**Les carnets du RAG — Master 2 Industry 4.0 / Systèmes informatiques pour le génie de la logistique industrielle et des services**

**Durée : 2 h.** Objectif : comprendre puis implémenter le passage **documents → chunks → embeddings → FAISS → recherche → évaluation**. Ce TP ne fait pas encore intervenir un LLM.

**Deux terrains d'application :** documentation de maintenance industrielle et procédures d'entrepôt / transport / service. Les douze documents et leurs procédures sont **entièrement fictifs** ; aucun protocole de sécurité réel n'est enseigné dans cet exercice.

> Le premier chargement d'un modèle Sentence Transformers nécessite Internet, notamment dans Colab.

## Étape 0 — Préparer l'environnement (10 min)

Dans Colab : **Exécution > Exécuter tout**, ou lancer chaque cellule dans l'ordre pour comprendre les résultats. Aucune clé d'API n'est nécessaire pour ce TP.

In [ ]:
%pip -q install 'sentence-transformers>=3,<6' 'faiss-cpu>=1.8' numpy pandas matplotlib

## Étape 1 — Inspecter le corpus documentaire (15 min)

Chaque document possède une identité, un domaine, un titre et une description. Les identifiants serviront à évaluer la recherche : ce sont des **sources attendues**, et non des réponses générées.

In [ ]:
import json, textwrap, numpy as np, pandas as pd
DATA = json.loads(r'''{"documents": [{"id": "IND-01", "titre": "Ligne B — Alarme VB-04", "domaine": "Industrie 4.0", "theme": "maintenance", "texte": "La fiche de maintenance fictive PR-17 concerne une alarme de vibration VB-04 sur le convoyeur de la ligne B. La procédure documentaire demande de consigner l’heure de l’alarme, l’identifiant de la machine et la valeur mesurée. Le responsable habilité vérifie l’historique des alarmes et l’état du dernier entretien. Le document précise que toute intervention physique dépend des consignes de sécurité du site et de l’autorisation du responsable de maintenance. L’assistant documentaire ne doit pas remplacer cette validation."}, {"id": "IND-02", "titre": "Moteur M-12 — Analyse vibratoire", "domaine": "Industrie 4.0", "theme": "capteurs", "texte": "Le rapport fictif VIB-M12 étudie les vibrations du moteur M-12. Les séries temporelles sont collectées toutes les dix minutes et agrégées par heure pour les rapports. Le rapport distingue vibration moyenne, maximum et dispersion. Une augmentation persistante déclenche une inspection documentaire de l’historique, et non un diagnostic automatique de panne. Les mesures doivent être horodatées et associées à la version du capteur."}, {"id": "IND-03", "titre": "Capteur de température T-8 — Métrologie", "domaine": "Industrie 4.0", "theme": "qualité", "texte": "La procédure fictive MET-T8 traite la calibration du capteur de température T-8. Chaque enregistrement de calibration comporte une date, un technicien habilité, une valeur de référence et un écart constaté. Le registre conserve les versions successives au lieu d’écraser l’historique. Pour une mesure actuelle, l’agent logiciel doit interroger la base de mesures autorisée ; un corpus documentaire statique ne suffit pas."}, {"id": "IND-04", "titre": "TRS / OEE — Production", "domaine": "Industrie 4.0", "theme": "performance", "texte": "Le guide fictif PROD-04 définit le taux de rendement synthétique TRS, appelé OEE en anglais, comme le produit disponibilité × performance × qualité. Les trois facteurs sont des proportions comprises entre zéro et un. Pour une disponibilité de 0,90, une performance de 0,95 et une qualité de 0,98, le TRS vaut environ 0,838. Le guide recommande de toujours mentionner la période analysée et la source des données."}, {"id": "IND-05", "titre": "Maintenance prédictive — Guide interne", "domaine": "Industrie 4.0", "theme": "maintenance", "texte": "Le guide fictif PREV-05 décrit une approche de maintenance prédictive utilisant vibration, température et historique des incidents. Le modèle statistique émet des scores de risque, qui ne doivent pas être présentés comme des certitudes. La validation doit vérifier les dérives de capteurs et les changements de conditions de fonctionnement. Les documents décrivent les modèles, tandis que les mesures récentes sont consultées dans un système de supervision."}, {"id": "IND-06", "titre": "Traçabilité des lots — Atelier", "domaine": "Industrie 4.0", "theme": "traçabilité", "texte": "Le document fictif TRACE-06 explique que chaque lot de production possède un identifiant unique. Les évènements de fabrication sont associés au lot, à une machine, à un créneau horaire et à une version de procédure. Lorsqu’une anomalie qualité est signalée, l’équipe cherche les lots concernés et la documentation applicable à la date concernée. Une réponse RAG doit citer la version du document plutôt que mélanger plusieurs révisions."}, {"id": "LOG-01", "titre": "Réception des marchandises — Entrepôt", "domaine": "Logistique et services", "theme": "entrepôt", "texte": "La procédure fictive REC-01 indique que toute réception doit être associée à un bon de livraison et à une commande attendue. En cas d’écart de quantité, l’agent enregistre une anomalie dans le WMS et la transmet au responsable de quai. Le stock n’est mis à jour qu’après la validation prévue par le processus interne. Les documents d’exploitation précisent les rôles ; l’état réel des commandes provient du WMS."}, {"id": "LOG-02", "titre": "Règle FEFO — Gestion des stocks", "domaine": "Logistique et services", "theme": "stocks", "texte": "Le référentiel fictif STOCK-02 présente la méthode FEFO : First Expired, First Out. Pour les produits soumis à date limite, les unités dont la date d’expiration est la plus proche sont préparées en priorité, sous réserve des contraintes de qualité et de disponibilité. Le texte distingue FEFO et FIFO, qui suit l’ordre d’entrée en stock. L’assistant doit citer la politique de stock effectivement applicable à la catégorie de produits."}, {"id": "LOG-03", "titre": "Quais de chargement — Ordonnancement", "domaine": "Logistique et services", "theme": "transport", "texte": "Le manuel fictif QUAI-03 décrit l’ordonnancement des quais de chargement. Chaque créneau est lié à un transporteur, à une unité de chargement et à un niveau de priorité. Lorsqu’un camion arrive en retard, la planification est ajustée par un agent habilité dans le TMS. Un RAG documentaire peut expliquer les règles du planning mais ne doit pas inventer l’horaire actuel d’un camion."}, {"id": "LOG-04", "titre": "Inventaire tournant — WMS", "domaine": "Logistique et services", "theme": "stocks", "texte": "La procédure fictive INV-04 présente l’inventaire tournant : une fraction des emplacements est contrôlée régulièrement sans immobiliser tout l’entrepôt. Les écarts entre stock physique et stock théorique sont enregistrés dans le WMS puis soumis à validation. Les comptages sont rattachés à une date, une zone et un opérateur autorisé. Les réponses fondées sur ce document décrivent la procédure mais pas les quantités réelles."}, {"id": "LOG-05", "titre": "Livraison — Gestion des incidents", "domaine": "Logistique et services", "theme": "services", "texte": "La politique fictive LIV-05 indique qu’un incident de livraison reçoit un numéro de ticket et une catégorie : retard, colis endommagé ou adresse incorrecte. Le service client consulte le TMS, vérifie la preuve de livraison lorsqu’elle existe et transmet l’incident au responsable concerné. Un modèle de langage peut expliquer les étapes du dossier, mais seul le système opérationnel donne le statut courant."}, {"id": "LOG-06", "titre": "Retours clients — Logistique inverse", "domaine": "Logistique et services", "theme": "services", "texte": "La procédure fictive RET-06 décrit les retours clients. Le retour est identifié par une autorisation, une référence de commande et un motif. La réception de l’article est distinguée de la décision de remise en stock. Une inspection qualité peut être nécessaire selon l’état du produit. La base documentaire décrit les conditions de traitement ; le statut du retour est maintenu dans le système transactionnel."}], "questions": [{"question": "Que prévoit le document PR-17 pour l’alarme VB-04 sur le convoyeur ?", "sources_attendues": ["IND-01"]}, {"question": "À quelle fréquence les mesures vibratoires du moteur M-12 sont-elles collectées ?", "sources_attendues": ["IND-02"]}, {"question": "Quels éléments sont consignés lors de la calibration du capteur T-8 ?", "sources_attendues": ["IND-03"]}, {"question": "Comment calcule-t-on le TRS ou OEE ?", "sources_attendues": ["IND-04"]}, {"question": "Quelles sources de données utilise la maintenance prédictive ?", "sources_attendues": ["IND-05"]}, {"question": "Comment la traçabilité des lots prend-elle en compte les versions des procédures ?", "sources_attendues": ["IND-06"]}, {"question": "Que faire lorsqu’une quantité reçue est différente de la commande ?", "sources_attendues": ["LOG-01"]}, {"question": "Quelle différence existe entre FEFO et FIFO ?", "sources_attendues": ["LOG-02"]}, {"question": "Comment est géré un camion en retard sur son créneau de quai ?", "sources_attendues": ["LOG-03"]}, {"question": "En quoi consiste un inventaire tournant ?", "sources_attendues": ["LOG-04"]}, {"question": "Comment catégoriser un incident de livraison dans le processus fictif ?", "sources_attendues": ["LOG-05"]}, {"question": "Quelle différence entre réception d’un retour et remise en stock ?", "sources_attendues": ["LOG-06"]}]}''')
documents, questions = DATA['documents'], DATA['questions']
print('Documents :', len(documents), '| questions :', len(questions))
display(pd.DataFrame(documents)[['id','titre','domaine','theme']])

**Question de réflexion.** Quelles différences existe-t-il entre un manuel de maintenance consultable dans un RAG et la valeur instantanée d'un capteur ? Une base documentaire suffit-elle à connaître un stock réel ?

## Étape 2 — Similarité cosinus « à la main » (20 min)

Calculer le cosinus de deux vecteurs permet de comprendre pourquoi on **normalise les embeddings** avant d'utiliser l'index FAISS `IndexFlatIP`.

In [ ]:
def cosinus(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))

u = [1, 2, 0]
v = [2, 4, 0]
w = [0, 0, 1]
print('cos(u,v)=', cosinus(u,v))
print('cos(u,w)=', cosinus(u,w))
assert np.isclose(cosinus(u,v), 1.0)
assert np.isclose(cosinus(u,w), 0.0)

**À discuter.** Un cosinus de 1 garantit-il que deux documents contiennent une information vraie ? Pourquoi non ?

## Étape 3 — Chunking et embeddings (30 min)

Notre découpage en mots est une **simplification pédagogique** : les systèmes de production comptent plutôt les tokens et préservent la structure des documents. Nous conservons l'ID source dans chaque chunk.

Modèle utilisé : `paraphrase-multilingual-MiniLM-L12-v2`. Les embeddings vectoriels pourront être comparés par similarité cosinus après normalisation.

In [ ]:
import re, faiss
from sentence_transformers import SentenceTransformer

def decouper(texte, taille=55, recouvrement=12):
    assert 0 <= recouvrement < taille
    mots=texte.split()
    pas=taille-recouvrement
    return [' '.join(mots[i:i+taille]) for i in range(0,len(mots),pas) if mots[i:i+taille]]

def construire_chunks(documents, taille=55, recouvrement=12):
    chunks=[]
    for doc in documents:
        for i, partie in enumerate(decouper(doc['texte'], taille, recouvrement)):
            chunks.append({'doc_id':doc['id'],'titre':doc['titre'],'domaine':doc['domaine'],
                           'chunk_num':i,'texte':partie})
    return chunks

chunks=construire_chunks(documents)
print(f'{len(chunks)} chunks pour {len(documents)} documents')
display(pd.DataFrame(chunks).head())

In [ ]:
MODELE = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encodeur=SentenceTransformer(MODELE)
textes=[c['titre'] + ' — ' + c['texte'] for c in chunks]
E=np.asarray(encodeur.encode(textes,normalize_embeddings=True,show_progress_bar=True),dtype='float32')
print('Matrice des embeddings :',E.shape)
print('Norme du premier vecteur :',float(np.linalg.norm(E[0])))

## Étape 4 — Index FAISS et requêtes (20 min)

Pour des vecteurs normalisés, le produit scalaire utilisé par `IndexFlatIP` correspond au cosinus. Chaque résultat reste associé au document dont il provient.

In [ ]:
def indexer(vecteurs):
    index=faiss.IndexFlatIP(vecteurs.shape[1])
    index.add(vecteurs)
    return index

index=indexer(E)

def chercher(question, top_k=4, index=index, chunks=chunks):
    q=np.asarray(encodeur.encode([question],normalize_embeddings=True),dtype='float32')
    scores, ids=index.search(q, min(top_k, index.ntotal))
    return [{**chunks[int(i)], 'score':float(s)} for s,i in zip(scores[0],ids[0]) if i >= 0]

def montrer(question,k=4):
    print('QUESTION :',question)
    for r in chercher(question,k):
        print(f"[{r['doc_id']}] score {r['score']:.3f} — {r['titre']}\n   {r['texte'][:180]}…")

montrer('Que faire en cas de quantité manquante lors d’une réception de marchandises ?')

**Exercice.** Testez une question sur le moteur M-12 puis une question sur FEFO. Pour chaque question, notez le premier résultat pertinent et les éventuels faux positifs.

## Étape 5 — Évaluer la récupération (15 min)

Chaque question a ici un document attendu. `Hit@k` vaut 1 lorsque ce document apparaît parmi les **k premiers chunks** récupérés et 0 sinon. Le `Recall@k` moyen coïncide, dans ce jeu d'essai à une seule source pertinente, avec la moyenne de `Hit@k`.

Cette évaluation teste la **récupération**, pas encore la fidélité d'une réponse générée.

In [ ]:
def hit_at_k(item,k=3, index=index, chunks=chunks):
    trouve={r['doc_id'] for r in chercher(item['question'],k,index,chunks)}
    attendu=set(item['sources_attendues'])
    return int(bool(trouve & attendu))

for k in [1,3,5]:
    score=np.mean([hit_at_k(q,k) for q in questions])
    print(f'Hit@{k} / moyenne Recall@{k} = {score:.1%}')

resultats=[]
for q in questions:
    titres=[r['doc_id'] for r in chercher(q['question'],3)]
    resultats.append({'question':q['question'], 'attendu':q['sources_attendues'][0],
                      'top_3':', '.join(titres),'succes':q['sources_attendues'][0] in titres})
display(pd.DataFrame(resultats))

## Étape 6 — Expérimenter puis interpréter (10 min)

**Travail à rendre :** comparer `(taille, recouvrement) = (30,8)`, `(55,12)` et `(90,15)`. Choisir deux questions présentant un classement différent et expliquer pourquoi les tailles de chunks affectent la recherche.

**Livrable :** notebook exécuté, tableaux comparatifs et court bilan distinguant les erreurs de retrieval des limites de la génération qui seront étudiées au TP 2.

In [ ]:
# À compléter par l'étudiant : comparer le nombre de chunks
for taille, rec in [(30,8),(55,12),(90,15)]:
    morceaux=construire_chunks(documents,taille,rec)
    print(f'taille={taille:>3}, overlap={rec:>2} -> {len(morceaux):>2} chunks')

# Défi : réencoder ces nouvelles configurations, reconstruire l'index FAISS
# et évaluer le Hit@3 sur les mêmes questions. Expliquez les écarts.

### Questions de synthèse

1. L'index contient-il les réponses ou des représentations numériques de passages ?
2. Pourquoi faut-il stocker les textes et les métadonnées en plus des vecteurs ?
3. Une similarité élevée prouve-t-elle la validité d'une procédure industrielle ?
4. Quelle différence y a-t-il entre retrouver un manuel de stock et lire le stock réel du WMS ?
5. Que faudra-t-il ajouter au TP 2 pour passer de la recherche à un assistant RAG ?

**Fin du TP 1.** Aucun modèle génératif n'est nécessaire pour cette première étape.